# 01 Data Check

## 1. Data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

#---------------設定整個專案資料夾路徑------------------------
ROOT = Path.cwd().resolve().parent
RAW_PATH = ROOT / "data" / "raw" / "YRBS_2007.csv"
PROCESSED_PATH = ROOT / "data" / "processed" / "yrbs_selected_cleaned.csv"
TAB_DIR = ROOT / "outputs" / "tables"
FIG_DIR = ROOT / "outputs" / "figures"
REF_DIR = ROOT / "references"
SUM_DIR = ROOT / "outputs" / "summary"

for p in [FIG_DIR, TAB_DIR, SUM_DIR, REF_DIR]:
    p.mkdir(parents=True, exist_ok=True)
#---------------設定整個專案資料夾路徑------------------------

#---------------所選的Behavior Variable for Proportion Analysis & Continuous Variable for Mean Analysis------------------------
BEHAVIOR_VAR = "EverCigaretteUse"
BEHAVIOR_P0 = 0.50
CONT_VAR = "HowMuchDoYouWeighWithoutShoesInKG"
CONT_MU0 = 68.0
#---------------所選的Behavior Variable for Proportion Analysis & Continuous Variable for Mean Analysis------------------------

#---------------根據上方BV和CV所需，處理Raw Data做成我們要的資料(Processed Data)並輸出成csv檔案-------------------
def ensure_processed():
    raw = pd.read_csv(RAW_PATH)
    behavior_raw = raw[BEHAVIOR_VAR]
    behavior_clean = behavior_raw.where(behavior_raw.isin([1, 2]))
    #behavior_binary欄位代表處理EverCigaretteUse資料的結果。Success=1;Failure=2;NaN=null -> Success=1;Failure=0;NaN=null
    behavior_binary = pd.Series(
        np.where(behavior_clean == 1, 1, np.where(behavior_clean == 2, 0, np.nan)),
        name="behavior_binary" #
    )
    cont_clean = pd.to_numeric(raw[CONT_VAR], errors="coerce")
    cont_clean = cont_clean.where(cont_clean > 0)
    processed = pd.DataFrame({
    BEHAVIOR_VAR: behavior_raw,
    "behavior_binary": behavior_binary,
    CONT_VAR: cont_clean,
    })
    processed.to_csv(PROCESSED_PATH, index=False)
    return raw, processed
#---------------根據上方BV和CV所需，處理Raw Data做成我們要的資料(Processed Data)並輸出成csv檔案-------------------

raw, processed = ensure_processed()
print("Raw data shape:", raw.shape)
print("Processed data shape:", processed.shape)

Raw data shape: (14041, 103)
Processed data shape: (14041, 3)


**Note:** 
- `behavior_binary` is created to recode the original behavior variable into a simpler binary format (**1 = success, 0 = failure**) so that the later EDA and one-sample proportion inference can be performed more clearly and consistently.

## 2. Research Questions

1. **Proportion question:** Is the proportion of students who have ever used cigarettes different from **0.50**?
2. **Mean question:** Is the mean body weight of students different from **68.0 kg**?

## 3. Variable Definition and Coding

### 3.1 Behavior Variable for Proportion Analysis
- Variable name: `EverCigaretteUse`
- What the variable measures: whether a student has ever used cigarettes
- valid codes used:
  - success = code 1
  - failure = code 2
- Recodeing used:
  - success = code 1
  - failure = code 0
- How missing or invalid values are handled: removed from the analysis
- Final valid sample size for behavior analysis: 13601

In [2]:
behavior_raw = raw[BEHAVIOR_VAR]

behavior_check = pd.DataFrame({
    "metric": [
        "Total rows",
        "Missing values",
        "Non-missing values",
        "Code 1 count",
        "Code 2 count",
        "Invalid non-missing values",
        "Final valid sample size"
    ],
    "value": [
        len(behavior_raw),
        int(behavior_raw.isna().sum()),
        int(behavior_raw.notna().sum()),
        int((behavior_raw == 1).sum()),
        int((behavior_raw == 2).sum()),
        int((behavior_raw.notna() & ~behavior_raw.isin([1, 2])).sum()),
        int(behavior_raw.isin([1, 2]).sum())
    ]
})

behavior_check.to_csv(TAB_DIR / "01_behavior_data_check.csv", index=False)
display(behavior_check)

behavior_raw_freq = (
    behavior_raw.value_counts(dropna=False)
    .sort_index()
    .rename_axis("raw_code")
    .reset_index(name="count")
)

behavior_raw_freq["raw_code"] = behavior_raw_freq["raw_code"].astype("object")
behavior_raw_freq["raw_code"] = behavior_raw_freq["raw_code"].where(
    behavior_raw_freq["raw_code"].notna(), "Missing"
)

behavior_raw_freq.to_csv(TAB_DIR / "01_behavior_raw_frequency.csv", index=False)

,metric,value
0,Total rows,14041
1,Missing values,440
2,Non-missing values,13601
3,Code 1 count,7164
4,Code 2 count,6437
5,Invalid non-missing values,0
6,Final valid sample size,13601


**Observations:** `EverCigaretteUse` contains only raw codes 1 and 2 plus missing values. 
This matches the assignment recoding rule, so no extra invalid codes need to be removed.

### 3.2 Continuous Variable for Mean Analysis
- Variable name: `HowMuchDoYouWeighWithoutShoesInKG`
- What the variable measures: body weight without shoes in kilograms
- Valid values used: positive numeric values
- How missing or invalid values are handled: removed from the analysis
- Final valid sample size for mean analysis: 13062

In [3]:
cont_original = raw[CONT_VAR]
cont_numeric = pd.to_numeric(cont_original, errors="coerce")

missing_count = int(cont_original.isna().sum())
invalid_non_missing_count = int(cont_original.notna().sum() - cont_numeric.notna().sum())
non_positive_count = int((cont_numeric.notna() & (cont_numeric <= 0)).sum())
final_valid_n = int((cont_numeric.notna() & (cont_numeric > 0)).sum())

continuous_check = pd.DataFrame({
    "metric": [
        "Total rows",
        "Missing values",
        "Non-missing values",
        "Non-positive values",
        "Invalid non-missing values",
        "Final valid sample size"
    ],
    "value": [
        len(cont_original),
        missing_count,
        int(cont_original.notna().sum()),
        non_positive_count,
        invalid_non_missing_count,
        final_valid_n
    ]
})

continuous_check.to_csv(TAB_DIR / "01_continuous_data_check.csv", index=False)
display(continuous_check)

,metric,value
0,Total rows,14041
1,Missing values,979
2,Non-missing values,13062
3,Non-positive values,0
4,Invalid non-missing values,0
5,Final valid sample size,13062


**Observations:** `HowMuchDoYouWeighWithoutShoesInKG` is numeric, has missing values, and has no non-positive values in the cleaned sample.
The analysis therefore keeps all positive recorded weights and removes missing values only.